#Environment setting

In [ ]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
SEED = 1   # 固定随机种子，每次运行结果都一样
print(ZONES)
print(TIME_BLOCKS)
print(COSTS)

['Central', 'North', 'Far West']
['Lunch', 'Weekday eve', 'Fri/Sat eve', 'Other']
{'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}


In [ ]:
times = delivery_times("Far West", "Fri/Sat eve", 45)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 42.9 minutes


# **Part1** How bad is it, really?

# **a**

##Late rate function

*   Definition of  late order: If its delivery time is longer than the promise time, the order is late.
*   I define two functions:
- `late_one(zone,time_block,promise)` looks at one zone and one time block. It returns the number of late orders and the number of all orders.
- `late_rate(zone,time_block,promise)` uses `late_one()`. If the zone or the time block is `'all'`, it checks every zone or every time block. It adds up the late orders and all orders, then divides them to get the late rate.




In [ ]:
# Block 1: count late orders and all orders for one zone-time block pair
def late_one(zone,time_block,promise):
  # get the delivery times of all orders in this pair
  times=delivery_times(zone,time_block,promise,seed=1)
  # put orders longer than the promise into the late list
  late=[]
  for t in times:
    if t>promise:
      late.append(t)
  # return the number of late orders and the number of all orders
  return len(late),len(times)

In [ ]:
# Test: Far West during Fri/Sat eve
n_late,n_orders=late_one("Far West","Fri/Sat eve",45)
print(f"In Far West during Fri/Sat eve, {n_late} of {n_orders} orders are late.")

In Far West during Fri/Sat eve, 77 of 210 orders are late.


In [ ]:
# Block 2: late rate function that also handles 'all'
def late_rate(zone,time_block,promise):
  # 'all' means every zone, otherwise only this one
  if zone=="all":
    zones=ZONES
  else:
    zones=[zone]

  if time_block=="all":
    blocks=TIME_BLOCKS
  else:
    blocks=[time_block]

  # add up late orders and all orders of every pair
  all_late=0
  all_orders=0
  for z in zones:
    for b in blocks:
      n_late,n_orders=late_one(z,b,promise)
      all_late=all_late+n_late
      all_orders=all_orders+n_orders
  # late rate = late orders / all orders * 100
  return round(all_late/all_orders*100,1)

## Test my function


In [ ]:
# 测试：Far West周五/六晚上
# Test: Far West during Fri/Sat eve
n_late,n_orders=late_one("Far West","Fri/Sat eve",45)
print(f"In Far West during Fri/Sat eve, {n_late} of {n_orders} orders are late.")

In Far West during Fri/Sat eve, 77 of 210 orders are late.


In [ ]:
# Case 2: all zones and a specific time block
zone="all"
time_block="Lunch"
promise=45
print(f"With a {promise}-minute promise, {late_rate(zone,time_block,promise)}% of orders in all zones during {time_block} are late.")

With a 45-minute promise, 3.3% of orders in all zones during Lunch are late.


In [ ]:
# Case 3: a specific zone and all time blocks
zone="Far West"
time_block="all"
promise=45
print(f"With a {promise}-minute promise, {late_rate(zone,time_block,promise)}% of orders in {zone} across all time blocks are late.")

With a 45-minute promise, 15.6% of orders in Far West across all time blocks are late.


In [ ]:
# Case 4: all zones and all time blocks
zone="all"
time_block="all"
promise=45
print(f"With a {promise}-minute promise, {late_rate(zone,time_block,promise)}% of all orders are late.")

With a 45-minute promise, 8.3% of all orders are late.


#**b**

## Ranking by late rates

By using `late_rate()` to get the late rate of all 12 zone-time block pairs, I put the late rate first in each tuple, so `sorted()` sorts the pairs by late rate. Then I sort them from high to low.

In [ ]:
# Late rate of all 12 zone-time block pairs (45-minute promise)
results=[]
for z in ZONES:
  for b in TIME_BLOCKS:
    rate=late_rate(z,b,45)
    # put the late rate first, so can sort by it
    results.append((rate,z,b))
print(f"There are {len(results)} zone-time block pairs.")

There are 12 zone-time block pairs.


In [ ]:
# Sort from the highest to the lowest
ranking=sorted(results,reverse=True)
rank=1
for row in ranking:
  print(f"Rank {rank}: in {row[1]} during {row[2]}, {row[0]}% of orders are late.")
  rank=rank+1

Rank 1: in Far West during Fri/Sat eve, 36.7% of orders are late.
Rank 2: in North during Fri/Sat eve, 16.7% of orders are late.
Rank 3: in Far West during Weekday eve, 11.2% of orders are late.
Rank 4: in Central during Fri/Sat eve, 6.8% of orders are late.
Rank 5: in North during Weekday eve, 6.0% of orders are late.
Rank 6: in Far West during Other, 4.7% of orders are late.
Rank 7: in Central during Weekday eve, 4.2% of orders are late.
Rank 8: in North during Lunch, 3.7% of orders are late.
Rank 9: in Central during Lunch, 3.5% of orders are late.
Rank 10: in Central during Other, 2.6% of orders are late.
Rank 11: in North during Other, 2.5% of orders are late.
Rank 12: in Far West during Lunch, 2.5% of orders are late.


Result: The biggest issue is orders from Far West area in Friday and Saturday evenings with a late-rate of 36.7%. Also, the top two pairs are both Fri/Sat evenings, so busy weekend evenings are the main problem.

#**c**

##Average delivery time






In [ ]:
# Block 1: total delivery minutes and number of orders for one zone-time block pair(same structure with a)
def minutes_one(zone,time_block,promise):
  # get the delivery times of all orders in this pair
  times=delivery_times(zone,time_block,promise,seed=1)
  # add up the delivery time of every order
  minutes=0
  for t in times:
    minutes=minutes+t
  # return the total minutes and the number of orders
  return minutes,len(times)

In [ ]:
# Test: Far West during Fri/Sat eve
n_minutes,n_orders=minutes_one("Far West","Fri/Sat eve",45)
print(f"In Far West during Fri/Sat eve, {n_orders} orders take {round(n_minutes,1)} minutes in total.")

In Far West during Fri/Sat eve, 210 orders take 9013.1 minutes in total.


In [ ]:
# Block 2: average delivery time function
def avg_delivery_time(zone,time_block,promise):
  # 'all' means every zone, otherwise only this one
  if zone=="all":
    zones=ZONES
  else:
    zones=[zone]

  if time_block=="all":
    blocks=TIME_BLOCKS
  else:
    blocks=[time_block]
  # add up minutes and orders of every pair
  all_minutes=0
  all_orders=0
  for z in zones:
    for b in blocks:
      n_minutes,n_orders=minutes_one(z,b,promise)
      all_minutes=all_minutes+n_minutes
      all_orders=all_orders+n_orders
  # average delivery time = all minutes / all orders
  return round(all_minutes/all_orders,1)

#Test the function

In [ ]:
# Case 1: a specific zone and a specific time block
zone="Far West"
time_block="Fri/Sat eve"
promise=45
print(f"With a {promise}-minute promise, the average delivery time in {zone} during {time_block} is {avg_delivery_time(zone,time_block,promise)} minutes.")

With a 45-minute promise, the average delivery time in Far West during Fri/Sat eve is 42.9 minutes.


In [ ]:
# Case 2: all zones and a specific time block
zone="all"
time_block="Lunch"
promise=45
print(f"With a {promise}-minute promise, the average delivery time in all zones during {time_block} is {avg_delivery_time(zone,time_block,promise)} minutes.")

With a 45-minute promise, the average delivery time in all zones during Lunch is 28.2 minutes.


In [ ]:
# Case 3: a specific zone and all time blocks
zone="Far West"
time_block="all"
promise=45
print(f"With a {promise}-minute promise, the average delivery time in {zone} across all time blocks is {avg_delivery_time(zone,time_block,promise)} minutes.")

With a 45-minute promise, the average delivery time in Far West across all time blocks is 35.3 minutes.


In [ ]:
# Case 4: all zones and all time blocks
zone="all"
time_block="all"
promise=45
print(f"With a {promise}-minute promise, the average delivery time of all orders is {avg_delivery_time(zone,time_block,promise)} minutes.")

With a 45-minute promise, the average delivery time of all orders is 31.2 minutes.


Result: None of the average delivery time is more than 45 mins.

#**d**

##Ranking by average delivery time (45-minute promise)
I use `avg_delivery_time()` to get the average delivery time of all 12 zone-time block pairs, then sort them from the longest to the shortest.

In [ ]:
# Average delivery time of all 12 zone-time block pairs
avg_results=[]
for z in ZONES:
  for b in TIME_BLOCKS:
    avg=avg_delivery_time(z,b,45)
    # put the average time first, so we can sort by it
    avg_results.append((avg,z,b))
print(f"There are {len(avg_results)} zone-time block pairs.")

There are 12 zone-time block pairs.


In [ ]:
# Sort from the longest average delivery time to the shortest
avg_ranking=sorted(avg_results,reverse=True)
rank=1
for row in avg_ranking:
  print(f"Rank {rank}: in {row[1]} during {row[2]}, the average delivery time is {row[0]} minutes.")
  rank=rank+1

Rank 1: in Far West during Fri/Sat eve, the average delivery time is 42.9 minutes.
Rank 2: in North during Fri/Sat eve, the average delivery time is 36.8 minutes.
Rank 3: in Far West during Weekday eve, the average delivery time is 35.0 minutes.
Rank 4: in Central during Fri/Sat eve, the average delivery time is 32.1 minutes.
Rank 5: in North during Weekday eve, the average delivery time is 31.4 minutes.
Rank 6: in Far West during Lunch, the average delivery time is 30.6 minutes.
Rank 7: in Far West during Other, the average delivery time is 30.0 minutes.
Rank 8: in Central during Weekday eve, the average delivery time is 29.0 minutes.
Rank 9: in North during Lunch, the average delivery time is 28.7 minutes.
Rank 10: in North during Other, the average delivery time is 27.7 minutes.
Rank 11: in Central during Lunch, the average delivery time is 26.8 minutes.
Rank 12: in Central during Other, the average delivery time is 26.0 minutes.


Results: All average delivery times are below the 45-minute promise, from 26.0 to 42.9 minutes.


#**e.** Late rate vs Average delivery time

To compare the two measures, I print them side by side for each pair, in the order of average delivery time.

In [ ]:
for row in avg_ranking:
  for r in results:
    if r[1]==row[1] and r[2]==row[2]:
      print(f"{row[1]} during {row[2]}: average {row[0]} minutes, late rate {r[0]}%")

Far West during Fri/Sat eve: average 42.9 minutes, late rate 36.7%
North during Fri/Sat eve: average 36.8 minutes, late rate 16.7%
Far West during Weekday eve: average 35.0 minutes, late rate 11.2%
Central during Fri/Sat eve: average 32.1 minutes, late rate 6.8%
North during Weekday eve: average 31.4 minutes, late rate 6.0%
Far West during Lunch: average 30.6 minutes, late rate 2.5%
Far West during Other: average 30.0 minutes, late rate 4.7%
Central during Weekday eve: average 29.0 minutes, late rate 4.2%
North during Lunch: average 28.7 minutes, late rate 3.7%
North during Other: average 27.7 minutes, late rate 2.5%
Central during Lunch: average 26.8 minutes, late rate 3.5%
Central during Other: average 26.0 minutes, late rate 2.6%


According the comparison above, the **late rate** in part 1b is more relevant to Rosa's decisions for several reasons:


1. **The average delivery time hides the problem.** All 12 averages are below the
45 minute promise. So we cann't address problems when looking only at the average. But 8.3% of all orders are late. For example, In Far West during Fri/Sat eve, the average is 42.9 minutes, still below 45, but 36.7% of orders are late.

2. **The two rankings do not agree.** Far West during Lunch has a longer average (30.6 minutes) than Far West during Other (30.0 minutes), but a lower late rate (2.5% vs 4.7%). If Rosa uses the average, she would focus on the wrong pair.
3. **The late rate is linked directly to the promise time**. When Rosa changes the promise time, the late rate changes directly.





#**Part 2** Exploring the best promise

#**a**
##Total cost of one late order
A late order costs Rosa in two ways (refund and potensial profits).

In [ ]:
# Total cost of one late order
def cost_per_late(costs):
  # direct cost: the refund given to the customer
  refund=costs["refund"]
  # churn cost: future orders lost x profit per order
  churn_cost=costs["churn_orders"]*costs["margin"]
  return refund+churn_cost

In [ ]:
# Test
cost=cost_per_late(COSTS)
print(f"Refund: ${COSTS['refund']:.2f}, lost future orders: {COSTS['churn_orders']}, margin per order: ${COSTS['margin']:.2f}")
print(f"One late order costs Rosa ${cost:.2f} in total.")
print(f"This equals the profit of about {round(cost/COSTS['margin'],1)} orders.")

Refund: $10.00, lost future orders: 1.8, margin per order: $9.00
One late order costs Rosa $26.20 in total.
This equals the profit of about 2.9 orders.


Avoiding one late order is worth giving up almost 3 orders. This trade-off is the key idea for choosing the best promise.

#**b**
##Choosing the best promise
For one zone and one time block over four weeks, the net profit is the profit from all orders minus the cost of late orders:

Net profit = number of orders × margin − number of late orders × cost per late order

A longer promise means fewer late orders, but also fewer customers. A shorter promise brings a few more customers, but many more late orders. The best promise is the one with the highest net profit.

In [ ]:
# Net profit of one zone-time block pair for one promise
def net_profit(zone,time_block,promise,costs):
  # number of late orders and all orders
  n_late,n_orders=late_one(zone,time_block,promise)
  # profit from all orders minus the cost of late orders
  profit=n_orders*costs["margin"]-n_late*cost_per_late(costs)
  return n_orders,n_late,profit

In [ ]:
# Test:
n_orders,n_late,profit=net_profit("Far West","Fri/Sat eve",45,COSTS)
print(f"With 45-minute promise, Far West on Fri/Sat eve has {n_orders} orders and {n_late} late orders, the net profit is {profit:.2f} dollars.")

With 45-minute promise, Far West on Fri/Sat eve has 210 orders and 77 late orders, the net profit is -127.40 dollars.


##Choosing the promise range
Before choosing the range, I try a very wide range, from 10 to 100 minutes, for Far West during Fri/Sat eve.

In [ ]:
# Try a wide range of promises (5 to 120 minutes) for Far West during Fri/Sat eve
for p in range(5,125,5):
  n_orders,n_late,profit=net_profit("Far West","Fri/Sat eve",p,COSTS)
  print(f"Promise {p} minutes: {n_orders} orders, {n_late} late, net profit {profit:.2f} dollars")

Promise 5 minutes: 242 orders, 242 late, net profit -4162.40 dollars
Promise 10 minutes: 242 orders, 242 late, net profit -4162.40 dollars
Promise 15 minutes: 241 orders, 241 late, net profit -4145.20 dollars
Promise 20 minutes: 241 orders, 239 late, net profit -4092.80 dollars
Promise 25 minutes: 239 orders, 233 late, net profit -3953.60 dollars
Promise 30 minutes: 237 orders, 218 late, net profit -3578.60 dollars
Promise 35 minutes: 232 orders, 179 late, net profit -2601.80 dollars
Promise 40 minutes: 224 orders, 133 late, net profit -1468.60 dollars
Promise 45 minutes: 210 orders, 77 late, net profit -127.40 dollars
Promise 50 minutes: 188 orders, 33 late, net profit 827.40 dollars
Promise 55 minutes: 158 orders, 8 late, net profit 1212.40 dollars
Promise 60 minutes: 121 orders, 2 late, net profit 1036.60 dollars
Promise 65 minutes: 84 orders, 0 late, net profit 756.00 dollars
Promise 70 minutes: 54 orders, 0 late, net profit 486.00 dollars
Promise 75 minutes: 32 orders, 0 late, net

The output above shows what happens with very short and very long promises.

If the promise is **very short**( from 5 to 25 mins), almost every order is late, and the net profit is very negative. Otherwise, if the promise is **very long**( above 60 minutes), late orders disappear, but customers stop ordering: 121 orders at 60 minutes, 32 at 75 minutes, and none at 110 minutes. The net profit falls to 0. Customers would also not accept such a long wait.

So I try promises **from 20 to 75 minutes** in steps of 5 minutes. This range covers the highest point of the net profit, with room on both sides.

In [ ]:
# Find the promise with the highest net profit
def best_promise(zone,time_block,promises,costs):
  # step 1: try every promise and save its net profit
  profits=[]
  for p in promises:
    n_orders,n_late,profit=net_profit(zone,time_block,p,costs)
    profits.append(profit)
  # step 2: find the highest net profit and its promise
  best_profit=max(profits)
  best_p=promises[profits.index(best_profit)]
  return best_p,best_profit

In [ ]:
# Promises:20, 25, ..., 75 minutes
promises=list(range(20,80,5))
print(promises)

[20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75]


#Test the function

In [ ]:
# Test: Far West during Fri/Sat eve
zone="Far West"
time_block="Fri/Sat eve"
best_p,best_profit=best_promise(zone,time_block,promises,COSTS)
# print the result of every promise, and mark the best one
for p in promises:
  n_orders,n_late,profit=net_profit(zone,time_block,p,COSTS)
  if p==best_p:
    print(f"Promise {p} minutes: {n_orders} orders, {n_late} late, net profit {profit:.2f} dollars !BSET!")
  else:
    print(f"Promise {p} minutes: {n_orders} orders, {n_late} late, net profit {profit:.2f} dollars")
n_orders,n_late,profit_45=net_profit(zone,time_block,45,COSTS)
print(f"For {zone} during {time_block}, the best promise is {best_p} minutes, with a net profit of {best_profit:.2f} dollars.")
print(f"With today's 45-minute promise, the net profit is only {profit_45:.2f} dollars.")
# check if the best promise is at the edge of the range
if best_p==promises[0] or best_p==promises[-1]:
  print("The best promise is at the edge of the range, so the range should be wider.")


Promise 20 minutes: 241 orders, 239 late, net profit -4092.80 dollars
Promise 25 minutes: 239 orders, 233 late, net profit -3953.60 dollars
Promise 30 minutes: 237 orders, 218 late, net profit -3578.60 dollars
Promise 35 minutes: 232 orders, 179 late, net profit -2601.80 dollars
Promise 40 minutes: 224 orders, 133 late, net profit -1468.60 dollars
Promise 45 minutes: 210 orders, 77 late, net profit -127.40 dollars
Promise 50 minutes: 188 orders, 33 late, net profit 827.40 dollars
Promise 55 minutes: 158 orders, 8 late, net profit 1212.40 dollars !BSET!
Promise 60 minutes: 121 orders, 2 late, net profit 1036.60 dollars
Promise 65 minutes: 84 orders, 0 late, net profit 756.00 dollars
Promise 70 minutes: 54 orders, 0 late, net profit 486.00 dollars
Promise 75 minutes: 32 orders, 0 late, net profit 288.00 dollars
For Far West during Fri/Sat eve, the best promise is 55 minutes, with a net profit of 1212.40 dollars.
With today's 45-minute promise, the net profit is only -127.40 dollars.
